Aggregate Silver data and create Gold fact and analytics tables.

In [0]:
from pyspark.sql import functions as F

items_per_order = (
    spark.table("ecommerce.silver.order_items")
    .groupBy("order_id")
    .agg(
        F.sum("price").alias("item_revenue"),
        F.sum("freight_value").alias("freight_revenue"),
        F.count("*").alias("item_count")
    )
)

payments_per_order = (
    spark.table("ecommerce.silver.payments")
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("payment_total"),
        F.count("*").alias("payment_count")
    )
)

display(items_per_order.limit(10))
display(payments_per_order.limit(10))


In [0]:

orders = spark.table("ecommerce.silver.orders")

gold_orders = (
    orders.alias("o")
    .join(
        items_per_order.alias("i"),
        on="order_id",
        how="left"
    )
    .join(
        payments_per_order.alias("p"),
        on="order_id",
        how="left"
    )
    .select(
        "order_id",
        "customer_id",
        "order_status",
        F.to_date("order_purchase_timestamp").alias("purchase_date"),
        F.coalesce(
            F.col("item_revenue"),
            F.lit(0).cast("decimal(12,2)")
        ).alias("item_revenue"),
        F.coalesce(
            F.col("freight_revenue"),
            F.lit(0).cast("decimal(12,2)")
        ).alias("freight_revenue"),
        F.coalesce(
            F.col("payment_total"),
            F.lit(0).cast("decimal(12,2)")
        ).alias("payment_total"),
        F.coalesce(F.col("item_count"), F.lit(0)).alias("item_count"),
        F.coalesce(F.col("payment_count"), F.lit(0)).alias("payment_count")
    )
)

(
    gold_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce.gold.fact_orders")
)

display(gold_orders.limit(10))


In [0]:

daily_sales = (
    spark.table("ecommerce.gold.fact_orders")
    .groupBy("purchase_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("item_revenue").alias("total_item_revenue"),
        F.sum("freight_revenue").alias("total_freight"),
        F.sum("payment_total").alias("total_payments"),
        F.avg("item_revenue").alias("avg_item_revenue_per_order")
    )
)

(
    daily_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce.gold.daily_sales")
)

display(
    spark.table("ecommerce.gold.daily_sales")
    .orderBy("purchase_date")
)


In [0]:
%sql

SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(item_revenue) AS total_item_revenue,
    SUM(freight_revenue) AS total_freight,
    SUM(payment_total) AS total_payments
FROM ecommerce.gold.fact_orders;


In [0]:
%sql

SELECT *
FROM ecommerce.gold.daily_sales
ORDER BY purchase_date DESC
LIMIT 30;
